# Phase 6: Texture Atlas Generation

**Pipeline position:**
```
Phase 1–5: Data through Mesh  ✓
Phase 6: Texture Mapping       ← YOU ARE HERE
Phase 7: Occlusion Inpainting
Phase 8: Evaluation
```

---

## Why Texture Mapping?

The mesh from Phase 5 is geometrically correct but **colourless** — a blank grey surface.  Texture mapping projects real photographic colour from the aerial images onto the mesh surface, producing a photorealistic 3-D model.

This is what transforms a reconstruction from a research prototype into something useful:
- **Urban digital twins** (city planning, property assessment)
- **Change detection** (compare textured before/after models)
- **Asset creation** (game engines, simulation environments)

---

## The Three-Step Texture Pipeline

```
For each triangle:
  1. VIEW SELECTION — which camera sees this face most directly?
  2. UV UNWRAPPING — map the 3-D triangle to a 2-D position in the atlas
  3. COLOUR PROJECTION — sample the selected camera's image at the UV coordinates
```

The result is a single **texture atlas** image (2048×2048 pixels) that contains the colour for every face of the mesh, packed together.

---

## Step 1: View Selection — Which Camera?

A mesh face can be photographed by multiple cameras.  We must pick the **best one** for texturing.  The score for camera $c$ viewing face $f$ is:

$$\text{score}(f, c) = \cos\theta = \frac{\mathbf{n}_f \cdot (\mathbf{e}_c - \mathbf{c}_f)}{\|\mathbf{e}_c - \mathbf{c}_f\|}$$

where:
- $\mathbf{n}_f$ = face normal (outward pointing)
- $\mathbf{e}_c$ = camera position (eye point)
- $\mathbf{c}_f$ = face centroid

A score near 1 means the camera is looking at the face **head-on** (optimal).  A score near 0 means the camera sees the face at a steep **oblique angle** — the face appears highly distorted in the image, giving poor texture quality.

We select the camera with the highest score for each face.

---

## Step 2: UV Unwrapping — The Atlas Concept

**UV coordinates** map every point on the 3-D surface to a 2-D position $(u, v) \in [0,1]^2$ in the texture image.

**UV unwrapping** is the process of "flattening" the 3-D mesh into this 2-D space.  For a complex mesh, you cannot flatten it without cuts — similar to peeling an orange and flattening the peel.

The mesh is cut into **UV islands** (connected patches that can be flattened without distortion), and these islands are **packed** into the atlas.

**Packing density** matters: wasted atlas space = lower effective texture resolution on the mesh.  Production tools like `xatlas` use bin-packing algorithms to maximise coverage.

---

## Step 3: Colour Projection

For each atlas texel, we need to fill it with the correct colour from the selected camera.  The process:

1. Find which mesh face covers this atlas texel (UV lookup)
2. Project the 3-D centroid of that face through the selected camera: $(u_{img}, v_{img}) = \pi(K[R|\mathbf{t}]\mathbf{c}_f)$
3. Sample the image at $(u_{img}, v_{img})$ (bilinear interpolation)
4. Write the colour to the atlas texel

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import cv2
import open3d as o3d
import matplotlib.pyplot as plt
import json
%matplotlib inline

In [ ]:
mesh_path  = os.path.join('..', 'outputs', 'mesh.ply')
poses_path = os.path.join('..', 'data', 'synthetic', 'GT_poses.json')
img_dir    = os.path.join('..', 'data', 'synthetic', 'images')
assert os.path.exists(mesh_path),  "Run phase5_mesh.ipynb first."
assert os.path.exists(poses_path), "Run phase1_data_generation.ipynb first."

mesh = o3d.io.read_triangle_mesh(mesh_path)
mesh.compute_vertex_normals()

with open(poses_path) as f:
    poses = json.load(f)

img_files = sorted(os.listdir(img_dir))
images    = [cv2.imread(os.path.join(img_dir, f)) for f in img_files]

print(f"Mesh     : {len(mesh.vertices):,} verts, {len(mesh.triangles):,} faces")
print(f"Cameras  : {len(poses)}")
print(f"Images   : {len(images)}  (shape: {images[0].shape})")

---

## Step 1 — Per-Face View Selection

We score every (face, camera) pair and select the camera with the highest cosine score.  The bar chart below shows how many faces are assigned to each camera — a balanced distribution indicates good camera coverage of the scene.

In [ ]:
from src.texture.view_selection import select_best_view

best_cam = select_best_view(mesh, poses)

valid_faces = (best_cam >= 0).sum()
print(f"Total faces     : {len(best_cam):,}")
print(f"Valid faces     : {valid_faces:,}  ({100*valid_faces/len(best_cam):.1f}%)")
print(f"Unassigned faces: {(best_cam < 0).sum():,}  (not visible from any camera)")

fig, ax = plt.subplots(figsize=(11, 4))
counts = [(best_cam == c).sum() for c in range(len(poses))]
bars = ax.bar(range(len(poses)), counts, color='steelblue', alpha=0.8)
ax.axhline(np.mean(counts), color='red', linestyle='--',
           label=f'mean = {np.mean(counts):.0f} faces/camera')
ax.set_xlabel('Camera ID'); ax.set_ylabel('Assigned faces')
ax.set_title('Face-to-Camera Assignment\nUniform distribution = good scene coverage')
ax.legend()
plt.tight_layout()
plt.show()

print("Camera 0 covers the most faces:"  if counts[0] == max(counts) else
      f"Camera {np.argmax(counts)} covers the most faces: {max(counts)} faces")

---

## Step 2 — UV Unwrapping

Each face gets a small region in the 2048×2048 atlas image.  The `generate_uv_atlas` function either uses `xatlas` (if installed) for angle-based flattening with good packing, or falls back to a simple shelf-packing approach.

The UV layout visualisation below shows how triangles are packed into the atlas space.  Good packing means more atlas coverage and thus higher effective texture resolution per triangle.

In [ ]:
from src.texture.uv_atlas import generate_uv_atlas

uvs, _ = generate_uv_atlas(mesh, atlas_size=2048)

print(f"UV array shape: {uvs.shape}  (faces × 3 vertices × 2 coordinates)")
print(f"UV value range: [{uvs.min():.4f}, {uvs.max():.4f}]  (should be in [0, 1])")

# Estimate packing efficiency: fraction of atlas used
n_sample = min(2000, len(uvs))
sample_idx = np.random.choice(len(uvs), n_sample, replace=False)

fig, ax = plt.subplots(figsize=(8, 8))
for fi in sample_idx:
    uv_px = uvs[fi] * 2048   # scale to pixel coordinates
    tri = plt.Polygon(uv_px, fill=False, edgecolor='steelblue', alpha=0.25, linewidth=0.3)
    ax.add_patch(tri)
ax.set_xlim(0, 2048); ax.set_ylim(0, 2048)
ax.set_title(f'UV Layout — {n_sample} sample faces\nin a 2048×2048 atlas')
ax.set_aspect('equal')
ax.set_xlabel('u (pixels)'); ax.set_ylabel('v (pixels)')
plt.tight_layout()
plt.show()

---

## Step 3 — Texture Projection and Atlas Building

For each face, we:
1. Project its centroid through the assigned camera to get image coordinates
2. Sample the image at those coordinates (bilinear interpolation)
3. Fill the corresponding UV region in the atlas with that colour

The atlas is then saved as a PNG.  A `.obj` file with a `.mtl` material definition references this PNG — the standard format for textured 3-D models compatible with Blender, Unity, Unreal Engine, and most 3-D viewers.

In [ ]:
from src.texture.texture_mapper import build_texture_atlas, save_textured_obj

print("Building texture atlas...")
atlas = build_texture_atlas(mesh, images, poses, atlas_size=2048)

out_dir = os.path.join('..', 'outputs')
os.makedirs(out_dir, exist_ok=True)
atlas_path = os.path.join(out_dir, 'texture_atlas.png')
cv2.imwrite(atlas_path, atlas)

coverage = (atlas.sum(axis=2) > 0).mean() * 100
print(f"Atlas saved    : {atlas_path}")
print(f"Atlas size     : {atlas.shape[1]}×{atlas.shape[0]} px")
print(f"Atlas coverage : {coverage:.1f}%  (filled pixels / total)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(cv2.cvtColor(atlas, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Texture Atlas  ({coverage:.1f}% coverage)\n'
                  f'Each patch = one UV island from the mesh')
axes[0].axis('off')

# Coverage mask
mask = (atlas.sum(axis=2) > 0).astype(np.uint8) * 255
axes[1].imshow(mask, cmap='Greys')
axes[1].set_title('Coverage Mask\nWhite = filled, Black = unfilled (will be inpainted in Phase 7)')
axes[1].axis('off')

plt.suptitle('Texture Atlas Analysis', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
obj_path = os.path.join(out_dir, 'textured_mesh.obj')
save_textured_obj(mesh, uvs, atlas_path, obj_path)
print(f"Textured mesh saved: {obj_path}")
print(f"Open with: Blender, MeshLab, or any 3-D viewer")

# Sample some face colours from the atlas
print("\nSampling 5 face colours from the atlas:")
for fi in np.random.choice(len(uvs), 5):
    uv_center = uvs[fi].mean(axis=0)  # UV centroid
    px = (uv_center * np.array([atlas.shape[1], atlas.shape[0]])).astype(int)
    px = np.clip(px, 0, [atlas.shape[1]-1, atlas.shape[0]-1])
    colour_bgr = atlas[px[1], px[0]]
    print(f"  Face {fi:5d}: UV=({uv_center[0]:.3f}, {uv_center[1]:.3f})  "
          f"BGR={colour_bgr}")

---

## Summary

| Step | Method | Output |
|---|---|---|
| View selection | Cosine score (face normal ⋅ camera direction) | best_cam[face] |
| UV unwrapping | xatlas / shelf packing | uvs: (F×3×2) |
| Colour projection | Project centroid → sample image | 2048×2048 atlas |
| Export | .obj + .mtl + .png | Loadable in any 3-D viewer |

### Questions to Think About

- The cosine score only considers viewing angle, not image resolution.  A camera far away but head-on could win over a close but slightly oblique camera.  How would you add distance into the score?
- If two cameras have very similar scores for the same face, their colours might differ slightly (different illumination angle).  This causes **texture seams** at face boundaries.  What strategies exist to reduce seams?
- Why do we use a single 2048×2048 atlas rather than one texture per face?  (Hint: think about GPU memory and draw calls.)
- What atlas coverage percentage would you expect for a scene with many vertical building walls?  (Hint: consider how many cameras have a good view of vertical faces.)

---

**Next → [Phase 7: Occlusion Inpainting](phase7_occlusion.ipynb)**